In [13]:
import os
import json
from docx import Document
from docx.shared import Pt
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

BASE_DIR = r"C:\Users\ivan\WORK\workshops\WWW\CFA\output_new"
OUTPUT_DIR = r"C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs"

# создаём папку для docx, если нет
os.makedirs(OUTPUT_DIR, exist_ok=True)

def highlight_run(run, color="FFFF99"):
    # добавляем бледно-жёлтую заливку
    shading_elm = OxmlElement('w:shd')
    shading_elm.set(qn('w:fill'), color)
    run._r.get_or_add_rPr().append(shading_elm)


def bold_before_colon(paragraph):
    """
    Если в параграфе есть ':', сделать жирным все до двоеточия.
    """
    text = paragraph.text
    if ':' in text:
        colon_index = text.index(':') + 1  # включительно
        # Удаляем старый текст
        paragraph.clear()
        # Добавляем жирный run для текста до :
        run_bold = paragraph.add_run(text[:colon_index])
        run_bold.bold = True
        # Добавляем обычный run для остального текста
        run_normal = paragraph.add_run(text[colon_index:])


for folder in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(folder_path):
        continue

    doc = Document()

    # 1. Abstract
    abstract_file_txt = os.path.join(folder_path, "abstract.txt")
    abstract_file_json = os.path.join(folder_path, "abstract.json")
    abstract_text = ""

    if os.path.exists(abstract_file_txt):
        with open(abstract_file_txt, "r", encoding="utf-8") as f:
            abstract_text = f.read()
    elif os.path.exists(abstract_file_json):
        with open(abstract_file_json, "r", encoding="utf-8") as f:
            data = json.load(f)
            abstract_text = data.get("abstract", "")

    if abstract_text:
        doc.add_heading("Abstract", level=1)
        doc.add_paragraph(abstract_text)

    # 2. Events / Entities
    events_file_json = os.path.join(folder_path, "entities_events.json")
    entities_events_text = ""
    
    if os.path.exists(events_file_json):
        with open(events_file_json, "r", encoding="utf-8") as f:
            data = json.load(f)
    
            # 1. Entities
            entities = data.get("entities", {})
            if entities:
                entities_lines = []
                for category, items in entities.items():
                    items_str = ", ".join(items)
                    entities_lines.append(f"{category.capitalize()}: {items_str}")
                entities_events_text += "Entities:\n" + "\n".join(entities_lines) + "\n\n"
    
            # 2. Events
            events_list = data.get("events", [])
            if events_list:
                events_lines = []
                for e in events_list:
                    type_ = e.get("type", "event")
                    desc = e.get("description", "")
                    events_lines.append(f"[{type_}] {desc}")
                entities_events_text += "Events:\n" + "\n".join(events_lines) + "\n\n"
    
            # 3. Relations
            relations_list = data.get("relations", [])
            if relations_list:
                relations_lines = []
                for r in relations_list:
                    cause = r.get("cause", "")
                    effect = r.get("effect", "")
                    relations_lines.append(f"Cause: {cause}\nEffect: {effect}")
                entities_events_text += "Relations:\n" + "\n\n".join(relations_lines)
                
    if entities_events_text:
        doc.add_heading("Entities / Events / Relations", level=1)
        doc.add_paragraph(entities_events_text)



    

    # 3. QA Counterfactual
    qa_counterfactual_file = os.path.join(folder_path, "qa_counterfactual.json")
    
    if os.path.exists(qa_counterfactual_file):
        with open(qa_counterfactual_file, "r", encoding="utf-8") as f:
            qa_data = json.load(f)
    
        doc.add_heading("QA Counterfactual", level=1)
        
        for i, qa in enumerate(qa_data, 1):
            doc.add_paragraph(f"Q{i}. Original Question: {qa.get('original_question', '')}")
            doc.add_paragraph(f"Counterfactual Question: {qa.get('counterfactual_question', '')}")
            doc.add_paragraph(f"Original Answer: {qa.get('original_answer', '')}")
            doc.add_paragraph(f"Counterfactual Answer: {qa.get('counterfactual_answer', '')}")
            doc.add_paragraph(f"Intervention: {qa.get('intervention', '')}")
            doc.add_paragraph("")  # пустая строка для разделения вопросов


    # --- После всех парсингов (entities/events/relations + QA) ---
    # Пройтись по всем параграфам документа
    # --- апостериорная построчная жирность до двоеточия ---
    apply_bold = False
    for p in doc.paragraphs:
        # включаем обработку после заголовка
        if p.text.strip().startswith("Entities / Events / Relations"):
            apply_bold = True
            continue
        if apply_bold:
            # разбиваем параграф на "строки" через переносы
            lines = p.text.splitlines()
            if len(lines) == 1:
                # обычная строка, без переносов
                if ':' in lines[0]:
                    colon_index = lines[0].index(':') + 1
                    paragraph_text = lines[0][colon_index:]
                    p.clear()
                    run_bold = p.add_run(lines[0][:colon_index])
                    run_bold.bold = True
                    p.add_run(paragraph_text)
            else:
                # есть переносы, каждый run делаем построчно
                p.clear()
                for line in lines:
                    if ':' in line:
                        colon_index = line.index(':') + 1
                        run_bold = p.add_run(line[:colon_index])
                        run_bold.bold = True
                        p.add_run(line[colon_index:] + "\n")
                    else:
                        p.add_run(line + "\n")



    # сохраняем docx в OUTPUT_DIR
    output_file = os.path.join(OUTPUT_DIR, f"{folder}_annotated.docx")
    doc.save(output_file)
    print(f"Создан {output_file}")


Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0000_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0001_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0002_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0003_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0004_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0005_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0006_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0007_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0008_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0009_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_0010_annotated.docx
Создан C:\Users\ivan\WORK\workshops\WWW\CFA\annotated_docs\paper_

In [6]:
from docx import Document

doc = Document()
doc.add_paragraph("Тестовый текст")
doc.save("test.docx")
